# SQL Course Flash Project: Employee Analytics & Department Performance

## Project Overview
This flash project demonstrates practical SQL skills through employee analytics and department performance analysis. It covers:
- Database design and normalization
- Complex queries with JOINs and aggregations
- Window functions and ranking
- Performance optimization
- Real-world business analysis

---

## 1. Database Schema Setup

### Create Tables

In [ ]:
-- Create Departments table
CREATE TABLE departments (
    dept_id INT PRIMARY KEY AUTO_INCREMENT,
    dept_name VARCHAR(100) NOT NULL UNIQUE,
    location VARCHAR(100),
    budget DECIMAL(12, 2),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- Create Employees table
CREATE TABLE employees (
    emp_id INT PRIMARY KEY AUTO_INCREMENT,
    emp_name VARCHAR(100) NOT NULL,
    dept_id INT NOT NULL,
    salary DECIMAL(10, 2),
    hire_date DATE,
    job_title VARCHAR(100),
    manager_id INT,
    performance_score DECIMAL(3, 2),
    FOREIGN KEY (dept_id) REFERENCES departments(dept_id),
    FOREIGN KEY (manager_id) REFERENCES employees(emp_id)
);

-- Create Projects table
CREATE TABLE projects (
    project_id INT PRIMARY KEY AUTO_INCREMENT,
    project_name VARCHAR(150) NOT NULL,
    dept_id INT NOT NULL,
    budget DECIMAL(12, 2),
    start_date DATE,
    end_date DATE,
    status VARCHAR(50),
    FOREIGN KEY (dept_id) REFERENCES departments(dept_id)
);

-- Create Employee_Projects assignment table
CREATE TABLE employee_projects (
    assignment_id INT PRIMARY KEY AUTO_INCREMENT,
    emp_id INT NOT NULL,
    project_id INT NOT NULL,
    hours_allocated INT,
    contribution_percentage DECIMAL(5, 2),
    FOREIGN KEY (emp_id) REFERENCES employees(emp_id),
    FOREIGN KEY (project_id) REFERENCES projects(project_id),
    UNIQUE KEY unique_emp_project (emp_id, project_id)
);

## 2. Sample Data Insert

### Populate with realistic data

In [ ]:
-- Insert Departments
INSERT INTO departments (dept_name, location, budget) VALUES
('Engineering', 'San Francisco', 500000.00),
('Sales', 'New York', 300000.00),
('HR', 'Chicago', 150000.00),
('Finance', 'Boston', 200000.00),
('Marketing', 'Los Angeles', 250000.00);

-- Insert Employees
INSERT INTO employees (emp_name, dept_id, salary, hire_date, job_title, manager_id, performance_score) VALUES
('Alice Johnson', 1, 120000.00, '2020-01-15', 'Senior Engineer', NULL, 4.8),
('Bob Smith', 1, 95000.00, '2021-03-20', 'Software Engineer', 1, 4.5),
('Carol Davis', 1, 85000.00, '2021-06-10', 'Junior Engineer', 1, 4.2),
('David Wilson', 2, 90000.00, '2020-05-05', 'Sales Manager', NULL, 4.6),
('Emma Brown', 2, 70000.00, '2022-01-10', 'Sales Rep', 4, 4.3),
('Frank Miller', 3, 75000.00, '2020-07-15', 'HR Manager', NULL, 4.4),
('Grace Lee', 4, 100000.00, '2019-11-01', 'Finance Manager', NULL, 4.7),
('Henry Taylor', 4, 65000.00, '2022-02-20', 'Accountant', 7, 4.1),
('Iris Martinez', 5, 85000.00, '2021-04-12', 'Marketing Manager', NULL, 4.5);

-- Insert Projects
INSERT INTO projects (project_name, dept_id, budget, start_date, end_date, status) VALUES
('Cloud Migration', 1, 150000.00, '2026-01-01', '2026-06-30', 'In Progress'),
('Mobile App', 1, 120000.00, '2026-02-01', '2026-08-31', 'In Progress'),
('Sales Dashboard', 2, 80000.00, '2026-03-01', '2026-05-31', 'Planning'),
('Q4 Campaign', 5, 100000.00, '2026-09-01', '2026-12-31', 'Planning'),
('ERP System', 4, 200000.00, '2025-06-01', '2027-06-30', 'In Progress');

-- Insert Employee-Project Assignments
INSERT INTO employee_projects (emp_id, project_id, hours_allocated, contribution_percentage) VALUES
(1, 1, 160, 80),
(1, 2, 80, 40),
(2, 1, 120, 60),
(2, 2, 120, 60),
(3, 2, 100, 50),
(4, 3, 80, 100),
(5, 3, 80, 100),
(7, 5, 120, 75),
(8, 5, 100, 50),
(9, 4, 160, 100);

## 3. Employee Analytics Queries

### Query 1: Employee Performance Rankings by Department

In [ ]:
-- Rank employees by performance within each department
SELECT 
    d.dept_name,
    e.emp_name,
    e.job_title,
    e.salary,
    e.performance_score,
    RANK() OVER (PARTITION BY d.dept_id ORDER BY e.performance_score DESC) AS dept_rank,
    RANK() OVER (ORDER BY e.performance_score DESC) AS company_rank
FROM employees e
JOIN departments d ON e.dept_id = d.dept_id
ORDER BY d.dept_name, dept_rank;

### Query 2: Department Salary Analysis

In [ ]:
-- Analyze salary distribution by department
SELECT 
    d.dept_name,
    COUNT(e.emp_id) AS employee_count,
    ROUND(AVG(e.salary), 2) AS avg_salary,
    MIN(e.salary) AS min_salary,
    MAX(e.salary) AS max_salary,
    ROUND(STDDEV(e.salary), 2) AS salary_stddev,
    SUM(e.salary) AS total_payroll,
    ROUND(SUM(e.salary) / d.budget * 100, 2) AS payroll_to_budget_pct
FROM employees e
JOIN departments d ON e.dept_id = d.dept_id
GROUP BY d.dept_id, d.dept_name, d.budget
ORDER BY avg_salary DESC;

### Query 3: Project Staffing Analysis

In [ ]:
-- Analyze project staffing and resource allocation
SELECT 
    p.project_name,
    p.status,
    COUNT(DISTINCT ep.emp_id) AS team_size,
    SUM(ep.hours_allocated) AS total_hours_allocated,
    ROUND(AVG(ep.contribution_percentage), 2) AS avg_contribution_pct,
    ROUND(SUM(e.salary) * SUM(ep.contribution_percentage) / 100 / 12, 2) AS monthly_labor_cost,
    DATEDIFF(p.end_date, p.start_date) AS project_duration_days
FROM projects p
LEFT JOIN employee_projects ep ON p.project_id = ep.project_id
LEFT JOIN employees e ON ep.emp_id = e.emp_id
GROUP BY p.project_id, p.project_name, p.status, p.start_date, p.end_date
ORDER BY total_hours_allocated DESC;

## 4. Department Performance Queries

### Query 4: Department Performance Scorecard

In [ ]:
-- Comprehensive department performance analysis
SELECT 
    d.dept_name,
    d.location,
    COUNT(DISTINCT e.emp_id) AS headcount,
    ROUND(AVG(e.performance_score), 2) AS avg_performance_score,
    COUNT(DISTINCT p.project_id) AS active_projects,
    SUM(ep.hours_allocated) AS total_allocated_hours,
    ROUND(SUM(e.salary) / 12, 2) AS monthly_payroll,
    CASE 
        WHEN AVG(e.performance_score) >= 4.6 THEN 'Excellent'
        WHEN AVG(e.performance_score) >= 4.3 THEN 'Good'
        WHEN AVG(e.performance_score) >= 4.0 THEN 'Satisfactory'
        ELSE 'Needs Improvement'
    END AS performance_rating
FROM departments d
LEFT JOIN employees e ON d.dept_id = e.dept_id
LEFT JOIN projects p ON d.dept_id = p.dept_id
LEFT JOIN employee_projects ep ON e.emp_id = ep.emp_id
GROUP BY d.dept_id, d.dept_name, d.location
ORDER BY avg_performance_score DESC;

### Query 5: Employee Utilization and Burnout Risk

In [ ]:
-- Identify overworked employees and utilization metrics
SELECT 
    e.emp_name,
    d.dept_name,
    COUNT(DISTINCT ep.project_id) AS num_projects,
    SUM(ep.hours_allocated) AS total_hours_allocated,
    ROUND(SUM(ep.contribution_percentage), 2) AS total_contribution_pct,
    e.performance_score,
    CASE 
        WHEN SUM(ep.contribution_percentage) > 100 THEN 'Over-allocated'
        WHEN SUM(ep.contribution_percentage) BETWEEN 80 AND 100 THEN 'Fully Utilized'
        WHEN SUM(ep.contribution_percentage) BETWEEN 50 AND 79 THEN 'Moderately Utilized'
        ELSE 'Under-utilized'
    END AS utilization_status,
    CASE 
        WHEN SUM(ep.contribution_percentage) > 100 AND e.performance_score < 4.3 THEN 'HIGH RISK'
        WHEN SUM(ep.contribution_percentage) > 100 THEN 'MEDIUM RISK'
        ELSE 'LOW RISK'
    END AS burnout_risk
FROM employees e
JOIN departments d ON e.dept_id = d.dept_id
LEFT JOIN employee_projects ep ON e.emp_id = ep.emp_id
GROUP BY e.emp_id, e.emp_name, d.dept_name, d.dept_id, e.performance_score
ORDER BY burnout_risk DESC, total_contribution_pct DESC;

### Query 6: Top Performers by Department with Career Progression

In [ ]:
-- Identify top performers and tenure
SELECT 
    d.dept_name,
    e.emp_name,
    e.job_title,
    e.salary,
    YEAR(CURDATE()) - YEAR(e.hire_date) AS years_employed,
    e.performance_score,
    ROUND(e.salary / LAG(e.salary, 1) OVER (PARTITION BY d.dept_id ORDER BY e.hire_date) * 100 - 100, 2) AS salary_increase_pct,
    DENSE_RANK() OVER (PARTITION BY d.dept_id ORDER BY e.performance_score DESC) AS dept_performance_rank
FROM employees e
JOIN departments d ON e.dept_id = d.dept_id
ORDER BY d.dept_name, dept_performance_rank
LIMIT 10;

## 5. Advanced Analysis

### Query 7: Department Efficiency Metrics

In [ ]:
-- Calculate department efficiency and ROI
WITH dept_stats AS (
    SELECT 
        d.dept_id,
        d.dept_name,
        COUNT(DISTINCT e.emp_id) AS headcount,
        SUM(e.salary) AS total_salary,
        AVG(e.performance_score) AS avg_performance,
        COUNT(DISTINCT p.project_id) AS project_count,
        SUM(CASE WHEN p.status = 'Completed' THEN 1 ELSE 0 END) AS completed_projects
    FROM departments d
    LEFT JOIN employees e ON d.dept_id = e.dept_id
    LEFT JOIN projects p ON d.dept_id = p.dept_id
    GROUP BY d.dept_id, d.dept_name
)
SELECT 
    dept_name,
    headcount,
    ROUND(total_salary / 12, 2) AS monthly_payroll,
    ROUND(total_salary / NULLIF(headcount, 0) / 12, 2) AS avg_monthly_salary_per_employee,
    project_count,
    completed_projects,
    ROUND(avg_performance, 2) AS avg_performance_score,
    ROUND(project_count / NULLIF(headcount, 0), 2) AS projects_per_employee,
    ROUND(completed_projects / NULLIF(project_count, 0) * 100, 2) AS completion_rate_pct
FROM dept_stats
ORDER BY avg_performance DESC;

## 6. Key Learnings & Takeaways

### SQL Concepts Demonstrated:
1. **JOINs**: Multiple INNER and LEFT JOINs for relational data
2. **Aggregation**: GROUP BY with COUNT, SUM, AVG, MIN, MAX
3. **Window Functions**: RANK(), DENSE_RANK(), ROW_NUMBER(), LAG()
4. **CTEs**: Common Table Expressions for complex queries
5. **CASE Statements**: Conditional logic for categorization
6. **Subqueries**: Nested queries for advanced filtering
7. **Performance Optimization**: Efficient use of indexes and aggregation

### Business Insights:
- Identify department performance trends
- Monitor employee utilization and burnout risk
- Track project profitability and resource efficiency
- Support data-driven HR and management decisions

---

**Project Type**: Flash Project | **Skill Level**: Intermediate | **Duration**: 2-3 hours